# Komoot — Surfaces par section (derniers 10/20/50 km)

Appelle directement `api.komoot.de/v007/tours/{id}/surfaces` et `/coordinates`
pour chaque course avec cobblestone ou gravel significatif (2018-2025).

**Output** : `race_data/surface_sections.csv`

In [28]:
import os, re, time, json
import numpy as np
import pandas as pd
import requests
import xml.etree.ElementTree as ET
from tqdm import tqdm

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from webdriver_manager.chrome import ChromeDriverManager

GPX_DIR    = '/Users/arthurdeletang/Desktop/Stage M1/Code/data/gpx_files_2'
MATCH_DIR  = '/Users/arthurdeletang/Desktop/Stage M1/Code/data/matching/all'
CHECKPOINT = '/Users/arthurdeletang/Desktop/Stage M1/Code/race_data/sections_checkpoint.json'
OUTPUT_CSV = '/Users/arthurdeletang/Desktop/Stage M1/Code/race_data/surface_sections.csv'

SECTIONS_KM  = [50, 20, 10]
COBBLE_CODES = {'sb#cobblestone', 'sb#paving_stones'}
GRAVEL_CODES = {'sb#compacted', 'sb#ground'}

API_HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36',
    'Host': 'api.komoot.de',
}

In [29]:
# ── Liste des courses a traiter ───────────────────────────────────────────────

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    d = np.radians([lat2-lat1, lon2-lon1])
    a = np.sin(d[0]/2)**2 + np.cos(np.radians(lat1))*np.cos(np.radians(lat2))*np.sin(d[1]/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

def distance_totale_gpx(filepath):
    try:
        root = ET.parse(filepath).getroot()
        ns = {'gpx': 'http://www.topografix.com/GPX/1/1'}
        pts = [(float(p.get('lat')), float(p.get('lon')))
               for p in root.findall('.//gpx:trkpt', ns)]
        if len(pts) < 2:
            return None
        return sum(haversine(pts[i][0], pts[i][1], pts[i+1][0], pts[i+1][1])
                   for i in range(len(pts)-1))
    except Exception:
        return None

def tour_id_from_url(url):
    m = re.search(r'/tour/(\d+)', str(url))
    return m.group(1) if m else None

df_komoot = pd.read_csv(os.path.join(MATCH_DIR, 'komoot_surface_all.csv'))
mask = (
    ((df_komoot['cobblestones_km'] > 0.5) | (df_komoot['compacted_gravel_km'] > 0.5)) &
    df_komoot['fichier_gpx'].str.match('^201[89]|^202', na=False) &
    df_komoot['route_url'].notna() & (df_komoot['route_url'] != '')
)
df_todo = df_komoot[mask][['fichier_gpx', 'route_url', 'cobblestones_km', 'compacted_gravel_km']].copy()

def find_gpx(fname):
    p = os.path.join(GPX_DIR, fname)
    if os.path.exists(p): return p
    p2 = os.path.join(GPX_DIR, fname.replace(' / ', ' - '))
    return p2 if os.path.exists(p2) else None

df_todo['gpx_path'] = df_todo['fichier_gpx'].apply(find_gpx)
df_todo['total_km'] = df_todo['gpx_path'].apply(lambda p: round(distance_totale_gpx(p), 2) if p else None)
df_todo['tour_id']  = df_todo['route_url'].apply(tour_id_from_url)
df_todo = df_todo[df_todo['total_km'].notna() & df_todo['tour_id'].notna() & (df_todo['total_km'] > 15)]

print(f'Courses a traiter : {len(df_todo)}')
print(df_todo[['fichier_gpx', 'tour_id', 'total_km', 'cobblestones_km']].head(5).to_string())

Courses a traiter : 1526
                                                                        fichier_gpx     tour_id  total_km  cobblestones_km
585  2018 11esimo Tr. Citta di S. Vendemiano - 58esimo GP Industria & Commercio.gpx  2969863792    173.52            1.190
588               2018 54esimo Giro della Regione Friuli Venezia Giulia Stage 2.gpx  2969866430    151.30            0.509
589               2018 54esimo Giro della Regione Friuli Venezia Giulia Stage 3.gpx  2969867494    158.79            0.761
590                                          2018 57esimo G.P Palio del Recioto.gpx  2969868496    156.40            3.060
600                                        2018 64esimo Trofeo Alcide Degasperi.gpx  2969877903    159.95            3.180


In [30]:
# ── Login Komoot via Selenium (une seule fois) ────────────────────────────────

options = Options()
options.add_argument('--window-size=1280,800')
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
driver.get('https://www.komoot.com/login')
print('Connecte-toi a Komoot, puis lance la cellule suivante.')

Connecte-toi a Komoot, puis lance la cellule suivante.


In [31]:
# ── Extraire les cookies vers une session requests ────────────────────────────
# Lance cette cellule APRES t'etre connecte.

driver.get('https://www.komoot.com/tour/2972108691')
WebDriverWait(driver, 25).until(
    lambda d: len(d.find_elements(By.CSS_SELECTOR, 'div.css-esdg3g')) > 0
)
time.sleep(2)

session = requests.Session()
for c in driver.get_cookies():
    session.cookies.set(c['name'], c['value'], domain=c.get('domain', '.komoot.de'))

driver.quit()

test = session.get(
    'https://api.komoot.de/v007/tours/2972108691/surfaces?format=v2&hl=en',
    headers=API_HEADERS, timeout=10
)
print(f'Test API surfaces -> {test.status_code}  (attendu : 200)')

Test API surfaces -> 200  (attendu : 200)


In [32]:
# ── Fonctions de calcul ───────────────────────────────────────────────────────

def fetch_json(tour_id, endpoint, timeout=25):
    url = f'https://api.komoot.de/v007/tours/{tour_id}/{endpoint}?format=v2&hl=en'
    try:
        r = session.get(url, headers=API_HEADERS, timeout=timeout)
        if r.status_code == 200:
            return r.json()
    except Exception:
        pass
    return None


def build_cum_dist(coord_items):
    lats = np.array([p['lat'] for p in coord_items], dtype=np.float64)
    lons = np.array([p.get('lng', p.get('lon', 0.0)) for p in coord_items], dtype=np.float64)
    R = 6371.0
    dlat = np.radians(np.diff(lats))
    dlon = np.radians(np.diff(lons))
    a = (np.sin(dlat/2)**2
         + np.cos(np.radians(lats[:-1])) * np.cos(np.radians(lats[1:])) * np.sin(dlon/2)**2)
    seg_km = R * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
    return np.concatenate([[0.0], np.cumsum(seg_km)])


def compute_sections(surf_items, cum_dist, sections_km=SECTIONS_KM):
    max_idx  = len(cum_dist) - 1
    total_km = cum_dist[-1]
    result = {}
    for n_km in sections_km:
        start_km = max(0.0, total_km - n_km)
        cob = 0.0
        grav = 0.0
        for it in surf_items:
            if it['element'] not in COBBLE_CODES and it['element'] not in GRAVEL_CODES:
                continue
            f = it['from']
            t = min(it['to'], max_idx)
            if f >= max_idx or t <= f:
                continue
            seg_start = cum_dist[f]
            seg_end   = cum_dist[t]
            if seg_end <= start_km:
                continue
            ov_km = min(seg_end, total_km) - max(seg_start, start_km)
            if ov_km <= 0:
                continue
            if it['element'] in COBBLE_CODES:
                cob  += ov_km
            else:
                grav += ov_km
        result[f'cobblestones_last_{n_km}km']     = round(cob,  3)
        result[f'compacted_gravel_last_{n_km}km'] = round(grav, 3)
    return result


def scrape_tour(tour_id):
    surf_data  = fetch_json(tour_id, 'surfaces')
    coord_data = fetch_json(tour_id, 'coordinates', timeout=30)
    if not surf_data or not coord_data:
        return None
    surf_items  = surf_data.get('items', [])
    coord_items = coord_data.get('items', [])
    if not surf_items or len(coord_items) < 2:
        return None
    cum_dist = build_cum_dist(coord_items)
    return compute_sections(surf_items, cum_dist)


print('Fonctions chargees.')

Fonctions chargees.


In [33]:
# ── Test Paris-Roubaix ────────────────────────────────────────────────────────

test_row = df_todo[df_todo['fichier_gpx'].str.contains('Paris-Roubaix', na=False)].head(1).iloc[0]
print(f'Course  : {test_row["fichier_gpx"]}')
print(f'tour_id : {test_row["tour_id"]}  | total_km : {test_row["total_km"]} km')
print(f'Komoot total cobblestones : {test_row["cobblestones_km"]} km')

res = scrape_tour(test_row['tour_id'])
print('\nResultat :')
for k, v in (res or {}).items():
    print(f'  {k}: {v} km')

Course  : 2019 Paris-Roubaix.gpx
tour_id : 2972108691  | total_km : 256.71 km
Komoot total cobblestones : 49.5 km

Resultat :
  cobblestones_last_50km: 14.386 km
  compacted_gravel_last_50km: 0.394 km
  cobblestones_last_20km: 6.351 km
  compacted_gravel_last_20km: 0.0 km
  cobblestones_last_10km: 1.499 km
  compacted_gravel_last_10km: 0.0 km


In [34]:
# ── Batch : 1526 courses ──────────────────────────────────────────────────────

def save_checkpoint(data):
    tmp = CHECKPOINT + '.tmp'
    with open(tmp, 'w') as f:
        json.dump(data, f, ensure_ascii=False)
    os.replace(tmp, CHECKPOINT)

results = {}
if os.path.exists(CHECKPOINT):
    with open(CHECKPOINT) as f:
        results = json.load(f)
    print(f'Checkpoint : {len(results)} deja traites')

todo = df_todo[~df_todo['fichier_gpx'].isin(results)]
print(f'Restant : {len(todo)}')

for _, row in tqdm(todo.iterrows(), total=len(todo)):
    fname = row['fichier_gpx']
    try:
        res = scrape_tour(row['tour_id'])
        results[fname] = res if res else {}
    except Exception as e:
        print(f'  Erreur {fname}: {e}')
        results[fname] = {}
    save_checkpoint(results)
    time.sleep(0.3)

print(f'Termine. {len(results)} courses traitees.')

Checkpoint : 14 deja traites
Restant : 1512


100%|██████████| 1512/1512 [13:55<00:00,  1.81it/s]

Termine. 1526 courses traitees.


In [37]:
# ── Export CSV ────────────────────────────────────────────────────────────────

rows = []
for fname, res in results.items():
    if not res:
        continue
    rows.append({'fichier_gpx': fname, **res})

df_out = pd.DataFrame(rows).fillna(0)
df_out.to_csv(OUTPUT_CSV, index=False)
print(f'Sauvegarde : {OUTPUT_CSV}  ({len(df_out)} lignes)')
print(df_out.sort_values('cobblestones_last_50km', ascending=False).head(20).to_string())
print(df_out.sort_values('compacted_gravel_last_50km', ascending=False).head(20).to_string())

Sauvegarde : /Users/arthurdeletang/Desktop/Stage M1/Code/race_data/surface_sections.csv  (1526 lignes)
                                       fichier_gpx  last_50km  last_20km  last_10km  cobblestones_last_50km  compacted_gravel_last_50km  cobblestones_last_20km  compacted_gravel_last_20km  cobblestones_last_10km  compacted_gravel_last_10km
165               2018 Tour de France Stage 21.gpx        0.0        0.0        0.0                  35.062                       0.000                  14.130                       0.000                   6.108                         0.0
377               2019 Tour de France Stage 21.gpx        0.0        0.0        0.0                  35.058                       0.000                  14.155                       0.000                   6.131                         0.0
853               2022 Tour de France Stage 21.gpx        0.0        0.0        0.0                  34.006                       0.000                  13.705                  